In [ ]:
####################################################
#### Gold-Standard-Skript ##########################
####################################################

import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
import joblib

# --- SCHRITT 1: Daten laden ---
# Der Wine-Datensatz ist ein Klassiker: 3 Weinsorten (Klassen), 13 chemische Merkmale
wine = load_wine()
X = wine.data
y = wine.target

print(f"Datensatz geladen: {X.shape[0]} Weine mit je {X.shape[1]} Merkmalen.")

# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split) ---
# WICHTIG: Wir zwacken VOR ALLEM anderen einen Test-Satz ab.
# Dieser wird niemals für Cross-Validation angefasst, sondern simuliert
# die "echte Welt" ganz am Ende.
# stratify=y sorgt dafür, dass die Verteilung der Weinsorten in beiden Sets gleich bleibt.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Die Pipeline bauen ---
# Das ist das Herzstück. Die Pipeline kapselt die Vorverarbeitung und das Modell.
pipeline = Pipeline([
    # a) Skalierung: Zwingend nötig für PCA
    ('scaler', StandardScaler()),
    
    # b) PCA: Reduktion auf 3 Hauptkomponenten (wie gewünscht)
    ('pca', PCA(n_components=3)),
    
    # c) Klassifikator: Logistische Regression
    ('classifier', LogisticRegression(random_state=42))
])

# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung) ---
# Wir nutzen StratifiedKFold, da wir eine Klassifikation haben.
# Das garantiert, dass in jedem Fold das Verhältnis der Weinsorten (Klassen) gleich bleibt.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# WICHTIG ZUR DATA LEAKAGE:
# cross_val_score übernimmt für uns die Arbeit.
# In jedem der 5 Durchläufe passiert intern folgendes:
# 1. Split in Training-Fold und Val-Fold.
# 2. scaler.fit() und pca.fit() NUR auf dem Training-Fold.
# 3. Transformation des Val-Folds mit den Parametern aus 2.
# -> Keine Data Leakage!
scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')

print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den 5 Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

# --- SCHRITT 5: Das finale Modell trainieren ---
# Wenn wir mit der CV zufrieden sind, trainieren wir die Pipeline
# nochmal auf den GESAMTEN Trainingsdaten (X_train), um so viele Daten wie möglich zu nutzen.
pipeline.fit(X_train, y_train)

# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
# Jetzt testen wir am unberührten Test-Set (X_test).
# Die Pipeline skaliert und transformiert X_test automatisch mit den Werten aus X_train.
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

# --- SCHRITT 7: Modell speichern (Deployment) ---
# Wir speichern die ganze Pipeline als eine Datei.
joblib.dump(pipeline, 'wine_model_pipeline.pkl')
print("\nModell wurde als 'wine_model_pipeline.pkl' gespeichert.")

# ---------------------------------------------------------
# --- Simulation: Später im Deployment (neues Skript) ---
print("\n--- Simulation: Nutzung im Deployment ---")

# 1. Laden
loaded_model = joblib.load('wine_model_pipeline.pkl')

# 2. Ein neuer, unbekannter Wein (Beispiel-Daten)
neuer_wein = X_test[0].reshape(1, -1) # Wir nehmen einfach den ersten aus dem Testset

# 3. Vorhersage
# Das Modell macht im Hintergrund automatisch: Skalierung -> PCA -> Regression
prediction = loaded_model.predict(neuer_wein)
print(f"Vorhergesagte Weinsorte: Klasse {prediction[0]}")
print(f"Tatsächliche Weinsorte:  Klasse {y_test[0]}")

In [ ]:
####################################################
#### Diamant-Standard-Skript #######################
####################################################
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone, BaseEstimator, ClassifierMixin
import joblib

# --- TEIL A: Die Ensemble-Klasse definieren ---
# Damit wir das später einfach speichern und laden können, bauen wir uns
# einen eigenen "Container", der die 5 Modelle hält.
class FoldEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, models):
        self.models = models # Liste der trainierten Pipelines

    def predict(self, X):
        # Ruft predict() für jedes Modell auf und macht eine Mehrheitsentscheidung (Hard Voting)
        # Oder einfacher: Wir nutzen direkt predict_proba für Soft Voting (besser)
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

    def predict_proba(self, X):
        # 1. Hole Wahrscheinlichkeiten von jedem einzelnen Modell
        # Ergebnis ist eine Liste von Arrays: [Modell1_Probas, Modell2_Probas, ...]
        all_probas = [model.predict_proba(X) for model in self.models]
        
        # 2. Bilde den Durchschnitt über alle Modelle (Soft Voting)
        avg_proba = np.mean(all_probas, axis=0)
        return avg_proba

# --- TEIL B: Datenvorbereitung ---
wine = load_wine()
X = wine.data
y = wine.target

# Hold-Out Split (Das Test-Set für ganz am Ende)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- TEIL C: Pipeline Blaupause ---
# Wir definieren die Architektur, aber fitten sie noch nicht.
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=3)),
    ('classifier', LogisticRegression(random_state=42))
])

# --- TEIL D: Manuelles K-Fold Training ---
n_folds = 5
cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

trained_models = []
fold_scores = []

print(f"Starte Training des {n_folds}-Fold Ensembles...\n")

# Wir iterieren manuell durch die Folds
for fold_idx, (train_index, val_index) in enumerate(cv.split(X_train_full, y_train_full)):
    # 1. Daten für diesen Fold holen
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]
    
    # 2. Eine frische Kopie der Pipeline erstellen (WICHTIG!)
    # clone sorgt dafür, dass wir wirklich bei Null anfangen (Random Init)
    current_model = clone(base_pipeline)
    
    # 3. Training auf dem Fold-Trainings-Teil
    current_model.fit(X_fold_train, y_fold_train)
    
    # 4. Validierung auf dem Fold-Validation-Teil (nur für unsere Info)
    score = current_model.score(X_fold_val, y_fold_val)
    fold_scores.append(score)
    
    # 5. Das trainierte Modell speichern! <--- DAS IST NEU
    trained_models.append(current_model)
    
    print(f"Fold {fold_idx+1}: Genauigkeit = {score:.2%}")

print("-" * 30)
print(f"Durchschnittliche CV-Genauigkeit der Einzelmodelle: {np.mean(fold_scores):.2%}")

# --- TEIL E: Erstellen des Ensembles ---
# Wir packen unsere 5 trainierten Pipelines in unsere Klasse
final_ensemble_model = FoldEnsemble(models=trained_models)

# --- TEIL F: Finale Evaluation am Test-Set ---
# Jetzt lassen wir das Ensemble auf die ungesehenen Test-Daten los.
# Das Ensemble nutzt intern alle 5 Modelle und mittelt die Vorhersage.
ensemble_score = np.mean(final_ensemble_model.predict(X_test) == y_test)
print(f"Ensemble Genauigkeit auf Test-Set: {ensemble_score:.2%}")

# --- TEIL G: Deployment (Speichern) ---
# Wir speichern das Ensemble-Objekt, das alle 5 Pipelines enthält.
joblib.dump(final_ensemble_model, 'wine_ensemble_pipeline.pkl')
print("\nEnsemble-Modell gespeichert.")

# ---------------------------------------------------------
# --- Simulation: Nutzung im Deployment ---
print("\n--- Simulation: Nutzung im Deployment ---")

loaded_ensemble = joblib.load('wine_ensemble_pipeline.pkl')
neuer_wein = X_test[0].reshape(1, -1)

# Hier passiert die Magie:
# 1. loaded_ensemble ruft 5x predict_proba auf (mit 5 verschiedenen Scalern/PCAs!)
# 2. Es mittelt die Ergebnisse
# 3. Es gibt die finale Klasse zurück
prediction = loaded_ensemble.predict(neuer_wein)
probabilities = loaded_ensemble.predict_proba(neuer_wein)

print(f"Vorhersage Ensemble: Klasse {prediction[0]}")
print(f"Wahrscheinlichkeiten (gemittelt): {probabilities[0]}")
print(f"Echte Klasse: {y_test[0]}")

In [ ]:
####################################################
#### Minimal Gold-Standard-Skript    ###############
####################################################

import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import joblib

# --- SCHRITT 1: Daten laden.
df = pd.read_csv("candorus_daten.csv")  # Pfad ggf. anpassen
y = df["Cluster"]
X = df.drop(columns=["Cluster"])

# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Die Pipeline bauen ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

# --- SCHRITT 5: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

# --- SCHRITT 7: Modell speichern (Deployment) ---
joblib.dump(pipeline, 'meinModell.pkl')
print("\nModell wurde als 'meinModell.pkl' gespeichert.")

In [ ]:
####################################################
#### Minimal Gold-Standard-Skript  Aepfel  ###############
####################################################

import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import joblib

# --- SCHRITT 1: Daten laden.
df = pd.read_csv("apple_quality.csv")  # Pfad ggf. anpassen
y = df["Quality"]
X = df.drop(columns=["Quality", "A_id"])

# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Die Pipeline bauen ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

# --- SCHRITT 5: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

# --- SCHRITT 7: Modell speichern (Deployment) ---
joblib.dump(pipeline, 'meinModell.pkl')
print("\nModell wurde als 'meinModell.pkl' gespeichert.")

In [ ]:
####################################################
#### Minimal Gold-Standard-Skript Heart Disease  ###############
####################################################

import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import joblib

# --- SCHRITT 1: Daten laden.
df = pd.read_csv("heart_failure_clinical_records_dataset.csv")  # Pfad ggf. anpassen
y = df["DEATH_EVENT"]
X = df.drop(columns=["DEATH_EVENT"])

# --- SCHRITT 2: Der "Goldene Schnitt" (Train-Test-Split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SCHRITT 3: Die Pipeline bauen ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# --- SCHRITT 4: K-Fold Cross Validation (Training & Validierung)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print("\n--- Cross-Validation Ergebnisse ---")
print(f"Genauigkeit in den Folds: {scores}")
print(f"Durchschnittliche Genauigkeit: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")

# --- SCHRITT 5: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

# --- SCHRITT 6: Finale Evaluation (Die Stunde der Wahrheit) ---
final_score = pipeline.score(X_test, y_test)
print(f"\nFinale Genauigkeit auf dem Test-Set: {final_score:.2%}")

# --- SCHRITT 7: Modell speichern (Deployment) ---
joblib.dump(pipeline, 'meinModell.pkl')
print("\nModell wurde als 'meinModell.pkl' gespeichert.")